# Phase 6 — Conformal Prediction (Colab T4)

**Pre-requisite:** Phase 5 done — robust model in Drive, calibration set (100 images)
carved in Phase 1, 25 corrupted sets available.

Conformal prediction produces a **set** of classes per object that contains the true
class with probability >= 1 - epsilon, with no distributional assumptions.

**Targets:** empirical coverage >= 0.95; avg set size clean <= 1.5; avg set size at
severity 5 > 2.0; coverage under corruption >= 0.90 even at severity 5.

In [ ]:
import os, json, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
SEVERITIES  = [1, 2, 3, 4, 5]
EPSILON = 0.05   # 5% error -> 95% coverage target

## Step 1 — Load data + model

In [ ]:
from ultralytics import RTDETR

LOCAL_DATA = '/content/robot_data'
if not os.path.exists(LOCAL_DATA):
    shutil.copytree(f'{PROJECT_DIR}/data/annotated', LOCAL_DATA)
CORRUPTED_LOCAL = '/content/corrupted'
if not os.path.exists(CORRUPTED_LOCAL) and os.path.exists(f'{PROJECT_DIR}/data/corrupted'):
    shutil.copytree(f'{PROJECT_DIR}/data/corrupted', CORRUPTED_LOCAL)

# Conformal prediction is post-hoc — no MC Dropout. Use the robust model.
model = RTDETR(f'{PROJECT_DIR}/models/robust/best.pt')

def _box_iou(a, b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter = max(0,x2-x1)*max(0,y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def load_gt(label_path, w, h):
    gt = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 5:
                    continue
                cls = int(p[0]); cx,cy,bw,bh = map(float, p[1:5])
                gt.append((cls, [(cx-bw/2)*w,(cy-bh/2)*h,(cx+bw/2)*w,(cy+bh/2)*h]))
    return gt

## Step 2 — Per-class probability proxy

RT-DETR gives per-query class scores, not a clean per-image softmax. Operational
definition used here: for a given GT box, `p_c` = the highest confidence among
predictions of class `c` that overlap the box (IoU >= 0.5); 0 if none. This is the
nonconformity proxy `s_i = 1 - p(true_class)`.

In [ ]:
def get_class_probs(model, img_path, gt_box, iou_thr=0.5):
    """Length-5 vector: per-class max confidence among preds overlapping gt_box."""
    res = model.predict(img_path, conf=0.001, verbose=False)[0]
    probs = np.zeros(len(CLASS_NAMES))
    if len(res.boxes) == 0:
        return probs
    boxes = res.boxes.xyxy.cpu().numpy()
    cls   = res.boxes.cls.cpu().numpy().astype(int)
    conf  = res.boxes.conf.cpu().numpy()
    for b, c, cf in zip(boxes, cls, conf):
        if _box_iou(b, gt_box) >= iou_thr and cf > probs[c]:
            probs[c] = cf
    return probs

## Step 3 — Nonconformity scores on the calibration set

In [ ]:
calib_img_dir = f'{LOCAL_DATA}/images/calibration'
calib_lbl_dir = f'{LOCAL_DATA}/labels/calibration'
calib_imgs = sorted(f'{calib_img_dir}/{f}' for f in os.listdir(calib_img_dir)
                    if f.lower().endswith(('.jpg','.jpeg','.png')))
print(f'Calibration images: {len(calib_imgs)}')

calibration_scores = []
for ip in calib_imgs:
    img = cv2.imread(ip); h, w = img.shape[:2]
    stem = os.path.splitext(os.path.basename(ip))[0]
    for true_cls, gt_box in load_gt(f'{calib_lbl_dir}/{stem}.txt', w, h):
        probs = get_class_probs(model, ip, gt_box)
        calibration_scores.append(1.0 - probs[true_cls])  # nonconformity

calibration_scores = np.array(calibration_scores)
print(f'Nonconformity scores: {len(calibration_scores)}')
print(f'  mean={calibration_scores.mean():.4f}  max={calibration_scores.max():.4f}')

## Step 4 — Conformal threshold

Exact quantile level mandated by the project: `q_level = ceil((n+1)(1-eps))/n`.

In [ ]:
n = len(calibration_scores)
q_level = np.ceil((n + 1) * (1 - EPSILON)) / n
q_level = min(q_level, 1.0)   # guard for tiny n
threshold = float(np.quantile(calibration_scores, q_level))

print(f'n (calibration scores): {n}')
print(f'q_level: {q_level:.4f}')
print(f'threshold: {threshold:.4f}')
# For n=100, eps=0.05: q_level = ceil(101*0.95)/100 = 96/100 = 0.96

## Step 5 — Prediction-set function

In [ ]:
def conformal_prediction_set(class_probs, threshold):
    """Include class c iff nonconformity (1 - p_c) <= threshold."""
    return [c for c, p in enumerate(class_probs) if (1.0 - p) <= threshold]

## Step 6 — Validate coverage on clean test set

In [ ]:
def evaluate_coverage(img_dir, lbl_dir, threshold):
    imgs = sorted(f'{img_dir}/{f}' for f in os.listdir(img_dir)
                  if f.lower().endswith(('.jpg','.jpeg','.png')))
    covered, total, sizes = 0, 0, []
    for ip in imgs:
        img = cv2.imread(ip); h, w = img.shape[:2]
        stem = os.path.splitext(os.path.basename(ip))[0]
        for true_cls, gt_box in load_gt(f'{lbl_dir}/{stem}.txt', w, h):
            probs = get_class_probs(model, ip, gt_box)
            pset = conformal_prediction_set(probs, threshold)
            if true_cls in pset:
                covered += 1
            sizes.append(len(pset))
            total += 1
    coverage = covered / total if total else 0.0
    return coverage, float(np.mean(sizes)) if sizes else 0.0, sizes

cov_clean, size_clean, sizes_clean = evaluate_coverage(
    f'{LOCAL_DATA}/images/test', f'{LOCAL_DATA}/labels/test', threshold)

print(f'Empirical coverage (clean test): {cov_clean:.4f}  (target >= {1-EPSILON:.2f})')
print(f'Avg prediction set size (clean): {size_clean:.4f}  (target <= 1.5)')
print('✅ coverage PASS' if cov_clean >= 1-EPSILON else '⚠️  coverage FAIL — recheck q_level formula')
print('✅ set size PASS' if size_clean <= 1.5 else '⚠️  set size FAIL')

## Step 7 — Coverage under corruption (worst corruption from Phase 3)

In [ ]:
# Pick the worst corruption recorded in Phase 3
with open(f'{PROJECT_DIR}/results/robustness_baseline.json') as f:
    base = json.load(f)
per_corruption = {c: np.mean([base['results_grid'][c][str(s)] for s in SEVERITIES])
                  for c in base['results_grid']}
worst = min(per_corruption, key=per_corruption.get)
print(f'Worst corruption (from Phase 3): {worst}')

all_sizes = {'clean': sizes_clean}
cov_by_sev, size_by_sev = {}, {}
for severity in SEVERITIES:
    sev_dir = f'{CORRUPTED_LOCAL}/{worst}/severity_{severity}'
    cov, size, sizes = evaluate_coverage(f'{sev_dir}/images', f'{sev_dir}/labels', threshold)
    cov_by_sev[severity] = cov
    size_by_sev[severity] = size
    all_sizes[f'S{severity}'] = sizes
    print(f'  severity {severity}: coverage={cov:.4f}  avg set size={size:.4f}')

print()
print(f'Avg set size at severity 5: {size_by_sev[5]:.4f}  (target > 2.0)')
print('✅ PASS' if size_by_sev[5] > 2.0 else '⚠️  FAIL')
print(f'Coverage at severity 5: {cov_by_sev[5]:.4f}  (target >= 0.90)')
print('✅ PASS' if cov_by_sev[5] >= 0.90 else '⚠️  FAIL')

## VIZ 6.A — Prediction set size distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = list(all_sizes.keys())
colors = plt.cm.RdYlGn(np.linspace(0.8, 0.2, len(labels)))
for label, color in zip(labels, colors):
    axes[0].hist(all_sizes[label], bins=range(1, len(CLASS_NAMES)+2),
                 alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_xlabel('Prediction set size'); axes[0].set_ylabel('Count')
axes[0].set_title('VIZ 6.A — Prediction set sizes'); axes[0].legend()

mean_sizes = [np.mean(all_sizes['clean'])] + [size_by_sev[s] for s in SEVERITIES]
x_labels = ['clean'] + [f'S{s}' for s in SEVERITIES]
axes[1].plot(x_labels, mean_sizes, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[1].axhline(y=1.0, color='green', linestyle='--', label='ideal (size=1)')
axes[1].set_xlabel('Corruption severity'); axes[1].set_ylabel('Mean set size')
axes[1].set_title('Mean set size grows with corruption'); axes[1].legend()

plt.suptitle('VIZ 6.A — Conformal prediction set sizes under corruption', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz6a_conformal_set_sizes.png', dpi=150)
plt.show()
print('⚠️  Clean should be mostly size 1; right plot should slope upward.')

## VIZ 6.B — Example prediction sets on real images

In [ ]:
import random
test_imgs = sorted(f'{LOCAL_DATA}/images/test/{f}'
                   for f in os.listdir(f'{LOCAL_DATA}/images/test')
                   if f.lower().endswith(('.jpg','.jpeg','.png')))
sample = random.sample(test_imgs, min(8, len(test_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, ip in zip(axes.flatten(), sample):
    img = cv2.imread(ip); h, w = img.shape[:2]
    stem = os.path.splitext(os.path.basename(ip))[0]
    gt = load_gt(f'{LOCAL_DATA}/labels/test/{stem}.txt', w, h)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if gt:
        true_cls, gt_box = gt[0]   # dominant object
        pset = conformal_prediction_set(get_class_probs(model, ip, gt_box), threshold)
        set_str = '{' + ', '.join(CLASS_NAMES[c] for c in pset) + '}'
        covered = true_cls in pset
        ax.set_title(f'True: {CLASS_NAMES[true_cls]}\nPred set: {set_str}',
                     fontsize=9, color='green' if covered else 'red')
    ax.axis('off')
plt.suptitle('VIZ 6.B — Conformal prediction sets (green=covered, red=missed)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz6b_conformal_examples.png', dpi=120)
plt.show()
print('⚠️  Red titles should be rare (< 5%).')

## Step 8 — Save metrics

In [ ]:
conformal_metrics = {
    'epsilon': EPSILON,
    'n_calibration': int(n),
    'q_level': float(q_level),
    'threshold': threshold,
    'coverage_clean': cov_clean,
    'avg_set_size_clean': size_clean,
    'worst_corruption': worst,
    'coverage_by_severity': {str(s): cov_by_sev[s] for s in SEVERITIES},
    'set_size_by_severity': {str(s): size_by_sev[s] for s in SEVERITIES},
}
with open(f'{PROJECT_DIR}/results/conformal_metrics.json', 'w') as f:
    json.dump(conformal_metrics, f, indent=2)
print('Saved results/conformal_metrics.json')

## Phase 6 Completion Checklist

- [ ] Calibration scores computed for all calibration images
- [ ] Threshold computed at epsilon = 0.05 (q_level printed; = 0.96 for n=100)
- [ ] Conformal prediction sets generated for all test images
- [ ] Empirical coverage >= 0.95 on clean test
- [ ] Avg set size clean <= 1.5; avg set size severity 5 > 2.0
- [ ] VIZ 6.A saved — set-size distribution + upward trend
- [ ] VIZ 6.B saved — example prediction sets, red titles < 5%
- [ ] `results/conformal_metrics.json` saved

Next → `07_analysis.ipynb`